# NB_PERF_03 · Comparar antes y después

**Paso final del framework de performance. Solo lectura**: compara dos corridas de `NB_PERF_01` + `NB_PERF_02` y escribe `informe_perf.md`, que va pegado en el PR.

Qué compara:

| Qué | Fuente | Regla |
|---|---|---|
| Duración de cada consulta (mediana del modo principal) | `mediciones_resumen.csv` | **Mejora** si baja más de `UMBRAL_MEJORA_PCT`; **regresión** si sube más de `UMBRAL_REGRESION_PCT` |
| Resultado de cada consulta | `huella` | Tiene que ser igual, salvo las consultas listadas en `CAMBIO_DE_RESULTADO_ESPERADO` |
| Memoria del modelo y de cada tabla | `vertipaq_modelo.csv`, `vertipaq_tablas.csv` | Informativo |
| Hallazgos del BPA por categoría | `bpa.csv` | Informativo |

**Veredicto**: `APROBADO` si no hay regresiones ni huellas distintas sin justificar y al menos una consulta mejora o la memoria baja más del umbral. Si no, `REVISAR`. El veredicto no mergea nada: lo decide una persona en el PR.

In [ ]:
# Parámetros
MODELO = "<nombre-del-modelo-semantico>"
CORRIDA_ANTES = "antes"
CORRIDA_DESPUES = "despues"
CARPETA_BASE = "/lakehouse/default/Files/doc_modelos"
MODO_PRINCIPAL = "frio"                  # frio refleja la primera vista de un usuario; caliente, la navegación
UMBRAL_MEJORA_PCT = 10
UMBRAL_REGRESION_PCT = 10
CAMBIO_DE_RESULTADO_ESPERADO = []        # ids de consultas donde la propuesta aprobada cambia el resultado a propósito
GUARDAR_DELTA = True

In [ ]:
# --- funciones (sin dependencias de Fabric: las prueba pruebas/prueba_local.py)
import pandas as pd


def _pct(antes, despues):
    if antes is None or pd.isna(antes) or antes == 0 or despues is None or pd.isna(despues):
        return None
    return round(100 * (despues - antes) / antes, 1)


def comparar_consultas(res_a: pd.DataFrame, res_d: pd.DataFrame, modo: str,
                       umbral_mejora: float, umbral_regresion: float,
                       cambio_esperado: list[str] | None = None) -> pd.DataFrame:
    cambio_esperado = set(cambio_esperado or [])
    a = res_a[res_a["modo"] == modo].set_index("id")
    d = res_d[res_d["modo"] == modo].set_index("id")
    filas = []
    for i in sorted(set(a.index) | set(d.index)):
        if i not in a.index or i not in d.index:
            filas.append({"id": i, "estado": "solo_en_" + ("antes" if i in a.index else "despues")})
            continue
        ra, rd = a.loc[i], d.loc[i]
        var = _pct(ra["total_ms_mediana"], rd["total_ms_mediana"])
        if ra["huella"] != rd["huella"] and i not in cambio_esperado:
            estado = "resultado_distinto"
        elif var is not None and var <= -umbral_mejora:
            estado = "mejora"
        elif var is not None and var >= umbral_regresion:
            estado = "regresion"
        else:
            estado = "igual"
        filas.append({
            "id": i, "estado": estado,
            "antes_ms": ra["total_ms_mediana"], "despues_ms": rd["total_ms_mediana"], "variacion_pct": var,
            "se_antes_ms": ra["se_ms_mediana"], "se_despues_ms": rd["se_ms_mediana"],
            "fe_antes_ms": ra["fe_ms_mediana"], "fe_despues_ms": rd["fe_ms_mediana"],
            "se_consultas_antes": ra["se_consultas_mediana"], "se_consultas_despues": rd["se_consultas_mediana"],
            "misma_huella": ra["huella"] == rd["huella"],
        })
    return pd.DataFrame(filas)


def comparar_memoria(tablas_a: pd.DataFrame, tablas_d: pd.DataFrame) -> pd.DataFrame:
    a = tablas_a.set_index("Table Name")["Total Size"]
    d = tablas_d.set_index("Table Name")["Total Size"]
    df = pd.concat([a.rename("antes_bytes"), d.rename("despues_bytes")], axis=1).reset_index()
    df = df.rename(columns={"index": "tabla", "Table Name": "tabla"})
    df["variacion_pct"] = [_pct(x, y) for x, y in zip(df["antes_bytes"], df["despues_bytes"])]
    return df.sort_values("antes_bytes", ascending=False, na_position="last")


def comparar_bpa(bpa_a: pd.DataFrame, bpa_d: pd.DataFrame) -> pd.DataFrame:
    a = bpa_a.groupby("Category").size().rename("antes")
    d = bpa_d.groupby("Category").size().rename("despues")
    df = pd.concat([a, d], axis=1).fillna(0).astype(int).reset_index().rename(columns={"index": "categoria", "Category": "categoria"})
    df["diferencia"] = df["despues"] - df["antes"]
    return df


def veredicto(cons: pd.DataFrame, bytes_antes: float, bytes_despues: float, umbral_mejora: float) -> tuple[str, list[str]]:
    motivos = []
    regresiones = cons[cons["estado"] == "regresion"]["id"].tolist()
    distintas = cons[cons["estado"] == "resultado_distinto"]["id"].tolist()
    if regresiones:
        motivos.append(f"Regresiones: {', '.join(regresiones)}")
    if distintas:
        motivos.append(f"Resultado distinto sin justificar: {', '.join(distintas)}")
    mejora_mem = _pct(bytes_antes, bytes_despues)
    hay_mejora = (cons["estado"] == "mejora").any() or (mejora_mem is not None and mejora_mem <= -umbral_mejora)
    if not hay_mejora:
        motivos.append("Ninguna consulta ni la memoria mejoran más que el umbral")
    return ("APROBADO" if not motivos else "REVISAR"), motivos


def _fmt_ms(v):
    return "-" if v is None or pd.isna(v) else f"{v:,.0f}"


def _fmt_pct(v):
    return "-" if v is None or pd.isna(v) else f"{v:+.1f} %"


def _fmt_mb(v):
    return "-" if v is None or pd.isna(v) else f"{v / 1024 ** 2:,.1f}"


def informe_md(modelo: str, corr_a: dict, corr_d: dict, modo: str, cons: pd.DataFrame,
               mem: pd.DataFrame, bpa: pd.DataFrame, bytes_a: float, bytes_d: float,
               resultado: str, motivos: list[str], nombres: dict | None = None) -> str:
    nombres = nombres or {}
    l = [f"## Performance: {modelo}", "",
         f"**Veredicto: {resultado}**" + (" · " + " · ".join(motivos) if motivos else ""), "",
         "| | Antes | Después |", "|---|---|---|",
         f"| Corrida | `{corr_a.get('corrida', '')}` | `{corr_d.get('corrida', '')}` |",
         f"| Commit | `{corr_a.get('commit', '') or '-'}` | `{corr_d.get('commit', '') or '-'}` |",
         f"| Medido (UTC) | {corr_a.get('mediciones_utc', '-')} | {corr_d.get('mediciones_utc', '-')} |",
         f"| Memoria del modelo (MB) | {_fmt_mb(bytes_a)} | {_fmt_mb(bytes_d)} ({_fmt_pct(_pct(bytes_a, bytes_d))}) |",
         "", f"### Consultas (mediana en modo {modo}, ms)", "",
         "| Consulta | Antes | Después | Variación | SE antes → después | FE antes → después | Estado |",
         "|---|---:|---:|---:|---|---|---|"]
    for _, r in cons.iterrows():
        nom = nombres.get(r["id"], "")
        etiqueta = f"{r['id']} {nom}".strip()
        if str(r["estado"]).startswith("solo_en_"):
            l.append(f"| {etiqueta} | | | | | | {r['estado']} |")
            continue
        l.append(f"| {etiqueta} | {_fmt_ms(r['antes_ms'])} | {_fmt_ms(r['despues_ms'])} | {_fmt_pct(r['variacion_pct'])} | "
                 f"{_fmt_ms(r['se_antes_ms'])} → {_fmt_ms(r['se_despues_ms'])} | "
                 f"{_fmt_ms(r['fe_antes_ms'])} → {_fmt_ms(r['fe_despues_ms'])} | {r['estado']} |")
    l += ["", "### Memoria por tabla (top 10, MB)", "", "| Tabla | Antes | Después | Variación |", "|---|---:|---:|---:|"]
    for _, r in mem.head(10).iterrows():
        l.append(f"| {r['tabla']} | {_fmt_mb(r['antes_bytes'])} | {_fmt_mb(r['despues_bytes'])} | {_fmt_pct(r['variacion_pct'])} |")
    l += ["", "### Best Practice Analyzer (hallazgos por categoría)", "", "| Categoría | Antes | Después | Diferencia |", "|---|---:|---:|---:|"]
    for _, r in bpa.iterrows():
        l.append(f"| {r['categoria']} | {r['antes']} | {r['despues']} | {r['diferencia']:+d} |")
    l += ["", "_Generado por NB_PERF_03. Las trazas son de uso exploratorio: comparar medianas, en la misma capacidad y en horarios parecidos._"]
    return "\n".join(l) + "\n"

## 1. Leer las dos corridas

In [ ]:
import json
import os
import re
from datetime import datetime, timezone

base = CARPETA_BASE.rstrip("/") + "/" + re.sub(r"[^\w\-]+", "_", MODELO) + "/perf"


def leer(corrida):
    c = f"{base}/{corrida}"
    faltan = [a for a in ["corrida.json", "mediciones_resumen.csv", "vertipaq_modelo.csv", "vertipaq_tablas.csv", "bpa.csv"]
              if not os.path.exists(f"{c}/{a}")]
    if faltan:
        raise FileNotFoundError(f"A la corrida '{corrida}' le faltan {faltan}: corré NB_PERF_01 y NB_PERF_02.")
    return {
        "corrida": json.load(open(f"{c}/corrida.json", encoding="utf-8")),
        "resumen": pd.read_csv(f"{c}/mediciones_resumen.csv", dtype={"id": str}),
        "modelo": pd.read_csv(f"{c}/vertipaq_modelo.csv"),
        "tablas": pd.read_csv(f"{c}/vertipaq_tablas.csv"),
        "bpa": pd.read_csv(f"{c}/bpa.csv"),
    }


A, D = leer(CORRIDA_ANTES), leer(CORRIDA_DESPUES)
consultas = pd.read_csv(f"{base}/consultas.csv", dtype=str, keep_default_na=False)
nombres = dict(zip(consultas["id"], consultas["nombre"]))

## 2. Comparar y escribir el informe

In [ ]:
cons = comparar_consultas(A["resumen"], D["resumen"], MODO_PRINCIPAL, UMBRAL_MEJORA_PCT,
                          UMBRAL_REGRESION_PCT, CAMBIO_DE_RESULTADO_ESPERADO)
mem = comparar_memoria(A["tablas"], D["tablas"])
bpa = comparar_bpa(A["bpa"], D["bpa"])
bytes_a, bytes_d = A["modelo"]["Total Size"].iloc[0], D["modelo"]["Total Size"].iloc[0]
resultado, motivos = veredicto(cons, bytes_a, bytes_d, UMBRAL_MEJORA_PCT)

md = informe_md(MODELO, A["corrida"], D["corrida"], MODO_PRINCIPAL, cons, mem, bpa,
                bytes_a, bytes_d, resultado, motivos, nombres)

salida = f"{base}/comparaciones/{CORRIDA_ANTES}__{CORRIDA_DESPUES}"
os.makedirs(salida, exist_ok=True)
open(f"{salida}/informe_perf.md", "w", encoding="utf-8").write(md)
cons.to_csv(f"{salida}/consultas.csv", index=False, encoding="utf-8")
mem.to_csv(f"{salida}/memoria.csv", index=False, encoding="utf-8")

if GUARDAR_DELTA:
    d = cons.astype(str)
    d["modelo"], d["corrida_antes"], d["corrida_despues"] = MODELO, CORRIDA_ANTES, CORRIDA_DESPUES
    d["veredicto"], d["fecha_utc"] = resultado, datetime.now(timezone.utc).isoformat(timespec="seconds")
    spark.createDataFrame(d).write.mode("append").option("mergeSchema", "true").saveAsTable("perf_comparacion")

print(md)
print(f"Guardado en {salida}/informe_perf.md: pegalo en el PR.")